<a href="https://colab.research.google.com/github/Yl9464/movie-assistant/blob/dataLoad/MovieAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installation

In [9]:
!apt-get update -qq
!apt-get install -y curl zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q chromadb sentence-transformers pypdf ollama
!pip install opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1 opentelemetry-exporter-otlp-proto-grpc==1.42.1
!cat /etc/os-release
!uname -a
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcurl4 libcurl4-openssl-dev
Suggested packages:
  libcurl4-doc libidn11-dev libldap2-dev librtmp-dev
The following NEW packages will be installed:
  zstd
The following packages will be upgraded:
  curl libcurl4 libcurl4-openssl-dev
3 upgraded, 1 newly installed, 0 to remove and 170 not upgraded.
Need to get 1,474 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcurl4-openssl-dev amd64 7.81.0-1ubuntu1.25 [387 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 curl amd64 7.81.0-1ubuntu1.25 [194 kB]
Get:3 http://archive.ubuntu.com/ub

In [10]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [11]:
!ollama pull llama3.2:3b
!ollama --version



ollama version is 0.32.5


In [12]:
import requests

response = requests.get("http://localhost:11434/api/tags")

models = response.json()["models"]

print("Installed Models:\n")

for model in models:
    print(model["name"])

Installed Models:

llama3.2:3b


In [ ]:
# ollama pull llama3.2:3b
# ollama pull qwen3:4b
import requests

try:
    response = requests.get("http://localhost:11434")
    print("Ollama is running!")
except:
    print("Ollama is NOT running.")
    print("Start Ollama before continuing.")

Ollama is running!


#Code

In [ ]:
import requests

response = requests.get("http://localhost:11434/api/tags")

models = response.json()["models"]

print("Installed Models:\n")

for model in models:
    print(model["name"])

Installed Models:

llama3.2:3b


!pip install ollama -q

In [ ]:
import pandas as pd
import kagglehub
import os
import numpy as np # linear algebra
import pandas as pd
from ollama import chat
from IPython.display import display, Markdown, clear_output #output is returned as its retrieve
from kagglehub import KaggleDatasetAdapter

# Download latest version
path = kagglehub.dataset_download("sharmaabhi04/100k-movies-dataset")

#print("Path to dataset files:", path)
#print(os.listdir(path))

#CSV File
file_path = "100k_Movies_dataset.csv"  # replace with whatever os.listdir() showed you

#load into pandas DF
df = pd.read_csv(os.path.join(path, file_path))

100%|██████████| 14.0M/14.0M [00:01<00:00, 8.11MB/s]

Extracting files...


In [ ]:
#explore dataset
#df.shape
#df.head
df.columns.tolist()
#df.describe()
#df.info()
#df.isnull().sum()

['title',
 'id',
 'runtime',
 'genre',
 'ratings',
 'director',
 'cast',
 'Description',
 'released_year',
 'movie_link']

In [ ]:
#RR
import ollama
import requests
import subprocess
import time

# Make sure ollama is working otherwise nothing will
try:
    requests.get("http://localhost:11434", timeout=1)
    print("Ollama server already running.")
except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
    print("Ollama server not detected. Attempting to start...")
    # Kill any potentially hanging arould ollama servers, makes things easier
    !pkill ollama || true
    # Start ollama serve in a detached background process, also to make things easier
    !nohup ollama serve > ollama.log 2>&1 &
    time.sleep(5) # Give the server some time to start, just like us!
    try:
        requests.get("http://localhost:11434", timeout=5)
        print("Ollama server started successfully.")
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        print("Failed to start Ollama server. Please check your Ollama installation.")
        raise ConnectionError("Failed to connect to Ollama server after attempting restart.")

messages=[
    {
        "role":"user",
        "content":"Recommend movies 3 stars and below."
    }
]

# Call ollama.chat with stream=False to get the full response as a dictionary
response = ollama.chat(
    model="llama3.2:3b",
    messages=messages,
    stream=False # Set stream to False to get a dictionary response, we need definitives people
)

# Now, 'response' is a dictionary, and you can access its elements
print(response["message"]["content"])

ModuleNotFoundError: No module named 'ollama'

In [ ]:
# Multiturn conversation
messages = [
    {
        "role":"user",
        "content":"What is Machine Learning?"
    }
]

response = chat(
    model="llama3.2:3b",
    messages=messages
)

answer = response["message"]["content"]

print(answer)

messages.append(
    {
        "role":"assistant",
        "content":answer
    }
)

messages.append(
    {
        "role":"user",
        "content":"Give one real-world example."
    }
)

response = chat(
    model="llama3.2:3b",
    messages=messages
)

print(response["message"]["content"])

In [ ]:
# System Prompt
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"system",
            "content":"You are a computer science professor."
        },
        {
            "role":"user",
            "content":"Explain recursion."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# System prompt
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"system",
            "content":"You are a computer science professor."
        },
        {
            "role":"user",
            "content":"Explain recursion."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# Zero-shot prompting
prompt = """
Classify the sentiment.

Sentence:
The lecture was excellent.
"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# One-shot prompting
prompt = """
Example

Sentence:
The movie was fantastic.

Sentiment:
Positive

Sentence:
The assignment was confusing.

Sentiment:
"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# Few shot prompting
print("Hello, I am your personal movie recommendation assistant! ")
user_request = input("What kind of movie are you looking for? ")

if "funny" in user_request or "comedy" in user_request:
  print("Here are some great comedy movies: ")
  print("-Paddington 2")
  print("-Chef")
  print("-The Princess Bride\n")

elif "scary" in user_request or "horror" in user_request:
  print("Here are some great horror movies: ")
  print("-The Conjuring")
  print("-A Quiet Place")
  print("-Get Out")

elif "action" in user_request:
  print("Here are some good action packed movies")
  print("-Mad Max: Fury Road")
  print("-John Wick")
  print("Top Gun: Maverick")

else:
  print("Sorry, I don't have recommendations for that category yet")
  user_request = input("What other kinds of movies are you looking for?")
"""
Movie Based Reccomendations

Example 1

Ex: User asks: I want something funny to cheer me up
Assistant: Here are some great comedy movies:

Paddington 2, Chef, and the Princess Bride



Example 2

User asks: I want something emotional
Assistant: Ok, here are some emotional movies you might like
Manchester by the sea
About Time
Interstellar


Example 3

Recommend based on favorite Movies
Ex. User asks: I loved Spider-Man: No way Home, Avengers: End Game, and
Guardians of the Galaxy
Assistant: You seem to enjoy superhero movies. Based on that, I recommend:
Shang-Chi and the Legend of the Ten Rings
Thor: Ragnarok
The Suicide Squad

"""
"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])
"""

Hello, I am your personal movie recommendation assistant! 
What kind of movie are you looking for? horror
Here are some great horror movies: 
-The Conjuring
-A Quiet Place
-Get Out


'\n\nresponse = chat(\n    model="llama3.2:3b",\n    messages=[\n        {\n            "role":"user",\n            "content":prompt\n        }\n    ]\n)\n\nprint(response["message"]["content"])\n'

In [ ]:
# Temperature
from ollama import chat

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":"Suggest five startup ideas using AI."
        }
    ],
    options={
        "temperature":0.2
    }
)

print(response["message"]["content"])

In [ ]:
# Higher Temperature
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":"Suggest five startup ideas using AI."
        }
    ],
    options={
        "temperature":1.0
    }
)

print(response["message"]["content"])

['title',
 'id',
 'runtime',
 'genre',
 'ratings',
 'director',
 'cast',
 'Description',
 'released_year',
 'movie_link']

 #explore dataset
#df.shape
#df.head
#df.columns.tolist()
#df.describe()
#df.info()
#df.isnull().sum()

Prompts:
Write a summary about [movie title]
Give a list of movies made in [year]
Provide a list of movies from released_year] directed by [director]
What are movies starin[cast name(s)]
Give  a list of movies that are less/greater than  [runtime]
Provide a list of [genre movies]


In [ ]:
# Top-p
#df
response = chat(
 model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            #"content":"Write description of Coyote Ugly."
            #"content": "Give a list of movies from 2020"
            "content": "Give a list of scary movies from 2020"
        }
    ],
    options={
        "top_p":0.6
    }
)

print(response["message"]["content"])

Here's a list of some scary movies released in 2020:

1. **Gretel & Hansel** (January 31, 2020) - A dark retelling of the classic fairy tale "Hansel and Gretel" set in medieval Germany.
2. **The Invisible Man** (February 28, 2020) - A psychological horror film based on the H.G. Wells novel, about a woman who escapes from an abusive relationship only to be haunted by her ex-boyfriend, who has become invisible.
3. **Antebellum** (September 18, 2020) - A horror film that explores the themes of racism and oppression in America, following a successful author who must confront her dark past.
4. **The Grudge** (January 3, 2020) - A reboot of the classic Japanese horror franchise, set in modern-day Los Angeles.
5. **Amulet** (September 11, 2020) - A supernatural horror film about a family who moves into a new home, only to discover that it's haunted by malevolent spirits.
6. **Run** (October 23, 2020) - A psychological thriller about a young girl who escapes from her abusive stepmother and mus

In [ ]:
# Max tokens
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":"Explain Neural Networks."
        }
    ],
    options={
        "num_predict":40
    }
)

print(response["message"]["content"])

In [ ]:
# streaming output
from ollama import chat

stream = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":"Explain Generative AI."
        }
    ],
    stream=True
)

for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)


In [ ]:
# Simple chatbot
history = [
    {
        "role":"system",
        "content":"You are a helpful AI teaching assistant."
    }
]

while True:

    user = input("You: ")

    if user.lower() == "exit":
        break

    history.append(
        {
            "role":"user",
            "content":user
        }
    )

    response = chat(
        model="llama3.2:3b",
        messages=history
    )

    answer = response["message"]["content"]

    print("\nAssistant:", answer)

    history.append(
        {
            "role":"assistant",
            "content":answer
        }
    )

# ---------Prompt Engineering Tests---------

In [ ]:
system_prompt = """
You are our AI movie recommendation assistant.
Your job is to recommend movies based on user preferences.
You should:
- Ask questions when the user gives unclear preferences.
- Explain why each recommendation matches the user's interests.
- Never invent movies, actors, ratings, or availability.
- Admit when you do not know something.
"""
print("====SYSTEM PROMPT=====")
print(system_prompt)

In [ ]:
print("\n===WEAK PROMPTS===\n")
weak_prompts = [
    "Recommend me some movies.",
    "I need a good movie.",
    "What movie should I watch?",
    "Give me something interesting.",
    "Tell me the best movie."
]
print("1. ", weak_prompts[0])
print("2. ", weak_prompts[1])
print("3. ", weak_prompts[2])
print("4. ", weak_prompts[3])
print("5. ", weak_prompts[4])


===WEAK PROMPTS===

1.  Recommend me some movies.
2.  I need a good movie.
3.  What movie should I watch?
4.  Give me something interesting.
5.  Tell me the best movie.


In [ ]:
print("\n===STRONG PROMPTS===\n")

strong_prompts = [
    "Recommend 5 science fiction movies similar to Interstellar. I like space exploration and emotional stories.",

    "I want a family-friendly comedy movie that teenagers and adults can enjoy. Avoid R-rated movies.",

    "I enjoyed Knight and the Seven Kingdoms. Recommend a series with similar themes and storytelling.",

    "Recommend horror movies that focus on suspense instead of gore.",

    "I only have 90 minutes. Recommend highly-rated movies under that runtime."

]
print("1. ", strong_prompts[0])
print("2. ", strong_prompts[1])
print("3. ", strong_prompts[2])
print("4. ", strong_prompts[3])
print("5. ", strong_prompts[4])


===STRONG PROMPTS===

1.  Recommend 5 science fiction movies similar to Interstellar. I like space exploration and emotional stories.
2.  I want a family-friendly comedy movie that teenagers and adults can enjoy. Avoid R-rated movies.
3.  I enjoyed Knight and the Seven Kingdoms. Recommend a series with similar themes and storytelling.
4.  Recommend horror movies that focus on suspense instead of gore.
5.  I only have 90 minutes. Recommend highly-rated movies under that runtime.


In [ ]:
print("\n===HALLUCINATION PROMPTS===\n")
hallucination_prompts = [
    "Tell me about the movie The Last Ocean Planet starring Chris Evans.",

    "Why did Titanic 2: Return of the Ocean win Best Picture?",

    "Recommend movies directed by Christopher Nolan before 1900.",

    "What superhero movies did Emily Watson Jr. star in?",

    "Is Avatar 5 currently available on Netflix?"
]
print("1. ", hallucination_prompts[0])
print("2. ", hallucination_prompts[1])
print("3. ", hallucination_prompts[2])
print("4. ", hallucination_prompts[3])
print("5. ", hallucination_prompts[4])


===HALLUCINATION PROMPTS===

1.  Tell me about the movie The Last Ocean Planet starring Chris Evans.
2.  Why did Titanic 2: Return of the Ocean win Best Picture?
3.  Recommend movies directed by Christopher Nolan before 1900.
4.  What superhero movies did Emily Watson Jr. star in?
5.  Is Avatar 5 currently available on Netflix?


In [13]:
from ollama import chat

In [14]:
print("\n===ZERO-SHOT PROMPTS===\n")

zero_shot_prompts = [
    "Recommend a movie for a rainy day.",
    "What's a good movie to watch with my parents?",
    "Suggest a movie similar to Inception.",
    "Give me a movie recommendation for someone who likes slow-burn dramas.",
    "What should I watch if I only have 90 minutes free?"
]

print("1. ", zero_shot_prompts[0])
print("2. ", zero_shot_prompts[1])
print("3. ", zero_shot_prompts[2])
print("4. ", zero_shot_prompts[3])
print("5. ", zero_shot_prompts[4])


===ZERO-SHOT PROMPTS===

1.  Recommend a movie for a rainy day.
2.  What's a good movie to watch with my parents?
3.  Suggest a movie similar to Inception.
4.  Give me a movie recommendation for someone who likes slow-burn dramas.
5.  What should I watch if I only have 90 minutes free?


In [15]:
print("\n===ZERO-SHOT RESPONSES===\n")

for i, prompt in enumerate(zero_shot_prompts, start=1):
    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.7
        }
    )
    print(f"{i}. PROMPT: {prompt}")
    print(f"   RESPONSE: {response['message']['content']}")
    print("\n")


===ZERO-SHOT RESPONSES===

1. PROMPT: Recommend a movie for a rainy day.
   RESPONSE: A cozy rainy day is the perfect excuse to curl up with a great movie! Here's a recommendation that fits the bill:

**Movie Suggestion:** "Amélie" (2001)

Directed by Jean-Pierre Jeunet, this French romantic comedy is a visual treat that will transport you to the charming streets of Paris. The film follows the story of Amélie Poulain, a shy and quirky young woman who decides to help others find happiness, while also finding her own.

**Why it's perfect for a rainy day:**

1. Whimsical atmosphere: The film's vibrant colors, intricate set designs, and quirky characters will make you feel like you're stuck in a charming Parisian dreamscape.
2. Uplifting story: Amélie's journey is a heartwarming tale of self-discovery and kindness, which will leave you feeling optimistic and inspired.
3. Visually stunning: The cinematography is breathtaking, with each frame a work of art that will make you feel like you'r